# VirtualMDT-Rx in Google Colab

**A virtual multidisciplinary team for safer prescribing.** Research demonstrator on invented data. **Not a medical device.** Do not enter real patient details.

Choose **Runtime > Run all**. The last cell prints a public link to the app (it ends in `gradio.live`) that works for 72 hours.

## 1. Get the code
This fetches the project from GitHub. If the repository is not public yet, the cell asks you to upload the project zip instead.

In [ ]:
import os, subprocess, sys
REPO = 'https://github.com/Muthuuk/VirtualMDT-Rx.git'
if not os.path.exists('VirtualMDT-Rx'):
    ok = subprocess.run(['git', 'clone', '--depth', '1', REPO]).returncode == 0
    if not ok:
        print('Could not fetch from GitHub. Please upload VirtualMDT-Rx.zip')
        from google.colab import files
        import zipfile, io
        up = files.upload()
        zipfile.ZipFile(io.BytesIO(next(iter(up.values())))).extractall('.')
%cd VirtualMDT-Rx
!pip -q install -r requirements.txt pytest

## 2. Check it before trusting it
First the 86 tests, then the ten acceptance targets. If any target fails, stop here.

In [ ]:
!python -m pytest -q
!python -m vmdt.evaluate

## 3. One case, step by step
A 78-year-old woman with reduced kidney function (eGFR 38). The request is nitrofurantoin for a urine infection.

In [ ]:
from vmdt.cases import PRESETS
from vmdt.orchestrator import run_case

case = PRESETS['Older woman, reduced kidney function, nitrofurantoin requested']
result = run_case(case)
print('OUTCOME :', result.outcome)
print('SUMMARY :', result.headline)
print('SAFER   :', result.alternative)
print()
for c in result.convened:
    print(f"  {c['name']:<18} {c['note']}")
print()
for f in result.findings:
    print(f"[{f.level:<5}] {f.title}\n         {f.detail}\n         ({f.agent}, rule {f.code}, source: {f.source})")

## 4. The 3D consulting room for that case
Hover over a figure to turn its card and read the reasons.

In [ ]:
from IPython.display import HTML
from vmdt import room3d
HTML(room3d.render(result))

## 5. Human in the loop
The case was escalated, so it cannot simply be approved. Watch the first attempt be refused.

In [ ]:
from vmdt.audit import AuditLog
from vmdt.orchestrator import sign_off

log = AuditLog()
print(sign_off(result, 'APPROVE', 'MR', 'GP', log)[1])
print(sign_off(result, 'ACCEPT_ALTERNATIVE', 'MR', 'GP', log)[1])
print('Log intact:', log.verify())

## 6. Review a whole list
200 invented patients on medicines for urine infection, type 2 diabetes and cancer care. The proportions are made up; this shows the method, not real epidemiology. To use your own anonymised CSV, replace `synthetic_cohort(200)` with `pd.read_csv('your_file.csv')`.

In [ ]:
import pandas as pd
from vmdt.batch import review_frame, synthetic_cohort

reviewed = review_frame(synthetic_cohort(200))
display(reviewed.groupby(['indication', 'outcome']).size().unstack(fill_value=0))
reviewed[reviewed.outcome != 'GREEN'].head(15)

## 7. What is actually being prescribed (live NHS open data)
Monthly items prescribed in English general practice, from OpenPrescribing.net. Change `'uti'` to `'type2_diabetes'` or `'cancer_care'`.

In [ ]:
from vmdt import opendata
df, message = opendata.condition_trends('uti')
print(message)
if not df.empty:
    display(opendata.summarise(df))
    ax = (df.pivot(index='date', columns='medicine', values='items') / 1000).plot(figsize=(11, 5), linewidth=2)
    ax.set_ylabel('Items per month (thousands)'); ax.set_ylim(bottom=0); ax.set_title('Urine infection antibiotics, England')

## 8. Launch the full app
Click the `gradio.live` link that appears.

In [ ]:
import app
app.launch(share=True, debug=False)